<a href="https://colab.research.google.com/github/jagadish-555/Personal_Finance_Anomaly_Detector-vortex/blob/main/final_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-learn joblib pandas matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

df.head()

In [ ]:
df.columns = df.columns.str.lower().str.strip()

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

df = df.dropna()
df = df.sort_values('date')

df.head()

In [ ]:

df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)


df = df.set_index('date')
df['tx_count_24h'] = df['amount'].rolling('24h').count()
df = df.reset_index()


df['user_running_avg'] = df['amount'].expanding().mean()
df['amt_to_user_avg_ratio'] = df['amount'] / (df['user_running_avg'] + 1e-9)

df.head()

In [ ]:

df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)


df = df.set_index('date')
df['tx_count_24h'] = df['amount'].rolling('24h').count()
df = df.reset_index()

# Behavioral deviation feature
df['user_running_avg'] = df['amount'].expanding().mean()
df['amt_to_user_avg_ratio'] = df['amount'] / (df['user_running_avg'] + 1e-9)

df.head()

In [ ]:
FEATURE_COLUMNS = [
    'amount',
    'hour',
    'day_of_week',
    'is_weekend',
    'tx_count_24h',
    'amt_to_user_avg_ratio'
]

X = df[FEATURE_COLUMNS].fillna(0)

X.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

In [ ]:
model = IsolationForest(
    n_estimators=150,
    contamination=0.05,
    random_state=42
)

model.fit(X_scaled)

In [ ]:
preds = model.predict(X_scaled)


df['anomaly'] = [1 if x == -1 else 0 for x in preds]

df.head()

In [ ]:
plt.figure(figsize=(12,6))

plt.scatter(df['date'], df['amount'], c=df['anomaly'], cmap='coolwarm')

plt.title("Transaction Anomaly Detection")
plt.xlabel("Date")
plt.ylabel("Amount")
plt.show()

In [ ]:
print("Total Transactions:", len(df))
print("Detected Anomalies:", df['anomaly'].sum())
print("Anomaly Percentage:", (df['anomaly'].sum()/len(df))*100)

In [ ]:
joblib.dump(model, "isolation_forest.pkl")
joblib.dump(scaler, "scaler.pkl")